In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Advanced Short-Answer Scoring with TF–IDF, Thai BERT, XGBoost, and Optuna Tuning
Author: Your Name

Description:
    1. Read train.csv, test.csv, sample_submission.csv
    2. Combine question + answer texts
    3. Compute TF–IDF features + Thai BERT embeddings
    4. Use Optuna to tune XGBoost hyperparameters with cross-validation
    5. Retrain the best model on the entire training set
    6. Predict on test.csv => submission.csv

Install:
    pip install scikit-learn xgboost transformers pythainlp tqdm optuna

Usage:
    python shortanswer_optuna.py
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import xgboost as xgb
import optuna


###############################################################################
# 1) Data I/O
###############################################################################
def read_data():
    """
    Reads train.csv, test.csv, and sample_submission.csv in the current directory.
    """
    train_path = os.path.join(REPO_PATH, "data", "train.csv")
    test_path = os.path.join(REPO_PATH, "data", "test.csv")
    sample_sub_path = os.path.join(REPO_PATH, "data", "sample_submission.csv")

    for p in [train_path, test_path, sample_sub_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sub_df = pd.read_csv(sample_sub_path)
    return train_df, test_df, sub_df


def combine_text(question, answer):
    """Join question + answer into one string."""
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    return q + " " + a


###############################################################################
# 2) Thai BERT Embedder
###############################################################################
class ThaiBERTEmbedder:
    """
    Encode text to embeddings using a Thai BERT (or multilingual) model from Hugging Face.
    """
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading BERT tokenizer/model = {model_name} on device = {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(self, text_list, batch_size=16, max_length=128):
        """
        Return shape: (num_texts, hidden_dim) array of BERT embeddings.
        """
        all_embs = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i : i + batch_size]
            inputs = self.tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)

            # [CLS] token embedding
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embs.append(cls_emb.cpu().numpy())

        return np.concatenate(all_embs, axis=0)


###############################################################################
# 3) Build TF–IDF
###############################################################################
def build_tfidf():
    """
    Create a TfidfVectorizer. Adjust parameters as desired.
    """
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,
        max_features=5000,
    )
    return vectorizer


###############################################################################
# 4) Precompute TF–IDF and BERT for train & test
###############################################################################
def preprocess_data(train_df, test_df, vectorizer, embedder):
    # Combine text
    train_df["text"] = train_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)
    test_df["text"]  = test_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)

    # TF–IDF
    print("Fitting TF–IDF on train data...")
    X_tfidf_train = vectorizer.fit_transform(train_df["text"].tolist())
    print("Transforming test data with TF–IDF...")
    X_tfidf_test  = vectorizer.transform(test_df["text"].tolist())

    # BERT embeddings
    print("Generating BERT embeddings (train)...")
    X_bert_train = embedder.encode(train_df["text"].tolist())
    print("Generating BERT embeddings (test)...")
    X_bert_test  = embedder.encode(test_df["text"].tolist())

    # Convert TF–IDF from sparse to dense for concatenation
    print("Converting TF–IDF to dense (may be memory-heavy for large data).")
    X_tfidf_train_arr = X_tfidf_train.toarray()
    X_tfidf_test_arr  = X_tfidf_test.toarray()

    # Concatenate horizontally
    X_train = np.hstack([X_tfidf_train_arr, X_bert_train])
    X_test  = np.hstack([X_tfidf_test_arr,  X_bert_test])

    y_train = train_df["score"].values
    return X_train, y_train, X_test


###############################################################################
# 5) Optuna: Tuning with Cross-Validation
###############################################################################
class OptunaTrainer:
    """
    Handle cross-validation with XGBoost inside an Optuna study to find best hyperparams.
    """
    def __init__(self, X, y, n_splits=3):
        self.X = X
        self.y = y
        self.n_splits = n_splits
        self.kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)

    def __call__(self, trial: optuna.trial.Trial):
        """
        Each Optuna trial proposes hyperparams => train XGB => return average MSE
        """
        # Suggest hyperparams
        n_estimators = trial.suggest_int("n_estimators", 100, 500)
        learning_rate = trial.suggest_float("learning_rate", 1e-4, 0.2, log=True)
        max_depth = trial.suggest_int("max_depth", 2, 12)
        subsample = trial.suggest_float("subsample", 0.6, 1.0)
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)

        # We can add more, e.g. 'min_child_weight', 'reg_alpha', 'reg_lambda', etc.

        mses = []
        for train_idx, val_idx in self.kf.split(self.X):
            X_train_fold, X_val_fold = self.X[train_idx], self.X[val_idx]
            y_train_fold, y_val_fold = self.y[train_idx], self.y[val_idx]

            model = xgb.XGBRegressor(
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                random_state=42,
                n_jobs=-1,
                tree_method="auto",  # or "gpu_hist" if GPU is available
                early_stopping_rounds=20,
            )
            model.fit(
                X_train_fold, y_train_fold,
                eval_set=[(X_val_fold, y_val_fold)],
                verbose=False,
            )
            y_pred_val = model.predict(X_val_fold)
            fold_mse = mean_squared_error(y_val_fold, y_pred_val)
            mses.append(fold_mse)

        mean_mse = float(np.mean(mses))
        return mean_mse


###############################################################################
# 6) Main script
###############################################################################
def main():
    print("=== (1) Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== (2) Building Tfidf & ThaiBERT ===")
    tfidf_vect = build_tfidf()
    embedder = ThaiBERTEmbedder(
        model_name="airesearch/wangchanberta-base-att-spm-uncased",
        device=None  # auto-detect GPU if present
    )

    print("=== (3) Preprocessing => TF–IDF + BERT embeddings ===")
    X_train, y_train, X_test = preprocess_data(train_df, test_df, tfidf_vect, embedder)

    print(f"Train shape = {X_train.shape}, Test shape = {X_test.shape}")

    print("=== (4) Hyperparameter Tuning with Optuna ===")
    # We'll do ~20 trials for demonstration (in practice 50+ is better)
    n_trials = 20
    study = optuna.create_study(direction="minimize")
    trainer = OptunaTrainer(X_train, y_train, n_splits=3)
    study.optimize(trainer, n_trials=n_trials, show_progress_bar=True)

    print("=== Best hyperparameters found ===")
    print(study.best_trial.params)
    best_params = study.best_trial.params

    print("=== (5) Training final model on full data with best hyperparams ===")
    final_model = xgb.XGBRegressor(
        **best_params,
        random_state=42,
        n_jobs=-1,
        tree_method="auto",
    )
    # We can disable early stopping if we want to use all n_estimators
    final_model.fit(X_train, y_train, verbose=True)

    print("=== (6) Predict on test set & Save submission ===")
    predictions = final_model.predict(X_test)
    sub_df["score"] = predictions
    sub_df.to_csv("submission.csv", index=False)
    print("Submission saved => submission.csv")


if __name__ == "__main__":
    main()


=== (1) Reading data ===
=== (2) Building Tfidf & ThaiBERT ===
Loading BERT tokenizer/model = airesearch/wangchanberta-base-att-spm-uncased on device = cpu
=== (3) Preprocessing => TF–IDF + BERT embeddings ===
Fitting TF–IDF on train data...
Transforming test data with TF–IDF...
Generating BERT embeddings (train)...
Generating BERT embeddings (test)...


[I 2025-03-07 17:54:06,688] A new study created in memory with name: no-name-2adddb0a-04f8-4c7a-8031-c1f4d16ec212


Converting TF–IDF to dense (may be memory-heavy for large data).
Train shape = (362, 3961), Test shape = (90, 3961)
=== (4) Hyperparameter Tuning with Optuna ===


  0%|          | 0/20 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[I 2025-03-07 17:54:52,077] Trial 0 finished with value: 2.711721506325708 and parameters: {'n_estimators': 400, 'learning_rate': 0.0021659776485012418, 'max_depth': 10, 'subsample': 0.8938172549719543, 'colsample_bytree': 0.9776498097323102}. Best is trial 0 with value: 2.711721506325708.
[I 2025-03-07 17:55:05,126] Trial 1 finished with value: 2.794644162932764 and parameters: {'n_estimators': 465, 'learning_rate': 0.002757464803748554, 'max_depth': 2, 'subsample': 0.7803106380338629, 'colsample_bytree': 0.9239963447580484}. Best is trial 0 with value: 2.711721506325708.
[I 2025-03-07 17:55:24,447] Trial 2 finished with value: 2.555397462121783 and parameters: {'n_estimators': 464, 'learning_rate': 0.0031258555911421806, 'max_depth': 4, 'subsample': 0.7595833887259633, 'colsample_bytree': 0.7788112525994363}. Best is trial 2 with value: 2.555397462121783.
[I 2025-03-07 17:55:41,303] Trial 3 finished with value: 2.529139936277192 and parameters: {'n_estimators': 238, 'learning_rate': 